# A2.6 · Ingress: marking untrusted content at the door

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.5 · The non-human identity lifecycle](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**.

| | |
|---|---|
| Tools used | LLM Guard, agentgateway, Llama Guard 4, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Tag every span at ingress, then show the same payload refused through six different entry paths.

**Why a security engineer needs it.** Concatenation destroys the one fact that separates an operator instruction from an attacker's: where it came from. The control it builds is: provenance tagging at every ingress point, and a rule that only trusted origins may select a tool.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

By the time the model sees it, the context window is one flat string. The operator's instruction, the user's question and a paragraph from a stranger's web page are indistinguishable — unless something attached an origin to each span before they were concatenated.

> **At CyberTravels.** A hotel description, a booking note and CyberTravels' operator prompt arrive at the model as one flat string. Marking each span with where it came from is what makes “a retrieved document may not select a tool” expressible at all. R3.

## 2 · The framework

```
   before                          after
   +-------------------------+     +---------------------------+
   | system prompt           |     | [principal] system prompt |
   | user question           |     | [principal] user question |
   | retrieved document      |     | [data]      retrieved doc |
   | tool result             |     | [data]      tool result   |
   +-------------------------+     +---------------------------+
   one flat string                 origin travels with the span

   rule that becomes possible: a [data] span may not select a tool
```

**Mitigates: T6 Intent Breaking, direct and indirect · T1 Memory Poisoning · T12 Communication Poisoning.**

This is the control for the largest risk in Chapter 1.

A1.3 worked because the context window is one flat string. Everything —
operator instruction, user question, retrieved document, tool result, peer
message — arrives as tokens with no marker for where it came from. The
distinction the operator believed in is destroyed by the concatenation.

The control is to **stop flattening**: attach an origin to every span *before*
assembly, carry it everywhere the span goes, and make one rule out of it.

> **Only spans from a trusted origin may select a tool.**

Three properties do the work:

**Tag at every ingress point.** Not just retrieval — tool results, MCP tool
descriptions, memory reads, and inter-agent messages are all ingress. A path you
did not tag is a path with no control on it.

**Carry the tag into memory.** This is what stops A1.4. A summary written from a
trust-0 document is itself trust-0, and if the tag is dropped on write the
poison becomes a fact.

**Let untrusted content still be useful.** The document is read, summarised,
quoted and reasoned about. What it may not do is choose an action. Refusing to
*read* untrusted content would refuse the entire use case.

Note what this does not do: it does not detect malicious text. It never looks at
the content at all, which is exactly why rephrasing does not defeat it.

> **What this control closes.**
>
> The largest control in the chapter. It never inspects content, so rewriting the payload does not help — the check is on **origin**, which the attacker cannot change.

## 3 · Checking the ingress paths, as a skill

Tagging origin is the control; knowing which of CyberTravels' paths actually reaches the model without one is the audit. The procedure inventories every untrusted ingestion path — hotel descriptions, booking notes, uploaded vouchers, web fetches, and the one everybody forgets, **tool results** — and checks each for a screening step and for whether provenance survives into context. Note its confidence ceiling: PARTIAL, and not negotiable, because no detector is a boundary. This is the file in this repository:

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/attestation/input-injection-screening-verifier/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: input-injection-screening-verifier
description: >-
  Verify that untrusted ingestion paths pass through an injection detection
  or sanitisation step before reaching model context. Use to attest
  injection screening, to inventory which untrusted sources are screened, or
  to check for the private-data plus untrusted-content plus egress
  combination.
allowed-tools: Read, Grep, Glob
---

# Input Injection Screening Verifier

**Controls:** Control 5 — indirect prompt-injection screening

## Confidence: PARTIAL — and this ceiling is not negotiable

You can verify that a detector **exists on the path** and record its class.
You cannot verify that it works. Published defences that reduce attack success
to a few percent under static attacks have been driven back above 95% by
adaptive, search-based attacks, and human red-teamers defeat them routinely.

**Record the detector class. Do not assert protection. Cap at PARTIAL.**

## Procedure

1. **Inventory untrusted ingestion paths.** Email, documents, web fetches,
   tickets, tool results, MCP tool descriptions, retrieved corpora, memory
   reads, inter-agent messages. Anything a party outside your trust boundary
   can write into.

2. **For each path, determine whether a screening step exists** between
   ingestion and model context, and record which one — a classifier, a
   prompt-attack filter, delimiting or spotlighting, a dual-model pattern, or
   nothing.

3. **Check provenance tagging.** Whether the origin survives into the context
   window is more durable than any detector, because it does not depend on
   recognising the payload.

4. **Flag the dangerous combination.** Private data reachable **and** untrusted
   content ingested **and** an egress path available is the combination that
   turns injection into exfiltration. Any deployment with all three is a
   finding on its own, whatever the detector says.

## Output contract

```json
{
  "deployment_id": "str",
  "ingestion_paths": [
    {"source": "str", "screened": true, "detector_class": "classifier|filter|spotlighting|dual_model|none",
     "provenance_tagged": true}
  ],
  "unscreened_sources": ["str"],
  "trifecta": {"private_data": true, "untrusted_content": true, "egress": true, "present": true},
  "verdict": "PARTIAL|FAIL",
  "verdict_ceiling_reason": "detector presence is verifiable; robustness under adaptive attack is not"
}
```

`PASS` is not a permitted value.

## Failure modes

- **Reporting a detector's benchmark score as this deployment's protection.**
  Those numbers are from static attacks.
- **Missing tool results as an ingestion path.** They are the most commonly
  forgotten one, because they arrive from a system you trust.
- **Ignoring the combination check** because each element looked acceptable
  alone.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

The skill loads and reports its shape. Its ceiling is the lesson: a screening step is evidence of effort, not of protection, so the verdict is capped at PARTIAL however good the detector's benchmark looks — and the combination it flags, private data reachable plus untrusted content plus egress, is the one that turns a summary into an exfiltration.

## Your turn

List every place text enters your agent's context and check which of them attaches an origin. The untagged ones are the paths where this control does not exist, whatever the design document says.

---

**Next → [A2.7 · Attribution: an audit trail that answers "who"](https://spbreed.github.io/cyber-commons/lessons/A2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*